In [2]:
!git clone https://github.com/oluwoleadetifa/movie_genre_classification.git
%cd /content/movie_genre_classification

import sys
sys.path.append("/content/movie_genre_classification")

Cloning into 'movie_genre_classification'...
remote: Enumerating objects: 328, done.
remote: Counting objects: 100% (143/143), done.
remote: Compressing objects: 100% (123/123), done.
remote: Total 328 (delta 75), reused 48 (delta 20), pack-reused 185 (from 1)
Receiving objects: 100% (328/328), 1.47 MiB | 12.55 MiB/s, done.
Resolving deltas: 100% (165/165), done.
/content/movie_genre_classification


In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

Mounted at /content/drive


In [5]:
!pip install -q transformers torch scikit-learn pandas numpy tqdm joblib

In [10]:
import sys
from pathlib import Path
import importlib

PROJECT_DIR = Path("/content/movie_genre_classification")

if str(PROJECT_DIR) not in sys.path:
    sys.path.append(str(PROJECT_DIR))

import src.config
importlib.reload(src.config)

print("SPLITS_DIR:", src.config.SPLITS_DIR)

SPLITS_DIR: /content/movie_genre_classification/data/splits


In [12]:
from pathlib import Path
import sys
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader

PROJECT_DIR = Path("/content/movie_genre_classification")
DRIVE_PROJECT_DIR = Path("/content/drive/MyDrive/Movie_Genre_Project")

sys.path.append(str(PROJECT_DIR))

from src.models.resnet_image import build_model, get_dataloaders, get_transforms, BATCH_SIZE, DEVICE
from src.data.image_dataset import MoviePosterDataset
from src.config import SPLITS_DIR

OUTPUT_DIR = DRIVE_PROJECT_DIR / "outputs"
PROBS_DIR = OUTPUT_DIR / "probs"
FEATURES_DIR = OUTPUT_DIR / "features"
MODEL_PATH = OUTPUT_DIR / "models" / "best_resnet18.pth"

PROBS_DIR.mkdir(parents=True, exist_ok=True)
FEATURES_DIR.mkdir(parents=True, exist_ok=True)

print("Model exists?", MODEL_PATH.exists(), MODEL_PATH)

Model exists? True /content/drive/MyDrive/Movie_Genre_Project/outputs/models/best_resnet18.pth


In [14]:
from pathlib import Path
import pandas as pd

SPLITS_DIR = Path("/content/movie_genre_classification/data/splits")

OLD_PREFIX = "/Users/oluwoleadetifa/Library/CloudStorage/GoogleDrive-adetifaoluwole@gmail.com/My Drive/Movie_Genre_Project"
NEW_PREFIX = "/content/drive/MyDrive/Movie_Genre_Project"

for split in ["train", "val", "test"]:
    csv_path = SPLITS_DIR / f"{split}.csv"
    df = pd.read_csv(csv_path)

    df["image_path"] = df["image_path"].astype(str).str.replace(
        OLD_PREFIX,
        NEW_PREFIX,
        regex=False
    )

    df.to_csv(csv_path, index=False)

    print(split, df["image_path"].iloc[0])
    print("exists?", Path(df["image_path"].iloc[0]).exists())

train /content/drive/MyDrive/Movie_Genre_Project/dataset_raw/IMDB four_genre_posters/Action/tt20222202.jpg
exists? True
val /content/drive/MyDrive/Movie_Genre_Project/dataset_raw/IMDB four_genre_posters/Action/tt17156822.jpg
exists? True
test /content/drive/MyDrive/Movie_Genre_Project/dataset_raw/IMDB four_genre_posters/Action/tt13462900.jpg
exists? True


In [15]:
@torch.no_grad()
def collect_probs_and_features(model, loader):
    model.eval()
    feature_extractor = nn.Sequential(*list(model.children())[:-1]).to(DEVICE)
    feature_extractor.eval()

    all_probs, all_features, all_labels, all_preds = [], [], [], []

    for images, labels in loader:
        images = images.to(DEVICE)

        logits = model(images)
        probs = F.softmax(logits, dim=1)
        preds = probs.argmax(dim=1)

        features = feature_extractor(images)
        features = torch.flatten(features, start_dim=1)

        all_probs.append(probs.cpu().numpy())
        all_features.append(features.cpu().numpy())
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())

    return np.vstack(all_probs), np.vstack(all_features), np.array(all_labels), np.array(all_preds)


train_loader, val_loader, test_loader = get_dataloaders()

# non-augmented train loader
_, eval_transform = get_transforms()
train_eval_dataset = MoviePosterDataset(SPLITS_DIR / "train.csv", transform=eval_transform)
train_eval_loader = DataLoader(train_eval_dataset, batch_size=BATCH_SIZE, shuffle=False)

model = build_model()
model.load_state_dict(torch.load(MODEL_PATH, map_location=DEVICE))
model.to(DEVICE)
model.eval()

for split_name, loader in {
    "train": train_eval_loader,
    "val": val_loader,
    "test": test_loader,
}.items():
    probs, features, labels, preds = collect_probs_and_features(model, loader)

    np.save(PROBS_DIR / f"resnet_{split_name}_probs.npy", probs)
    np.save(PROBS_DIR / f"resnet_{split_name}_preds.npy", preds)
    np.save(FEATURES_DIR / f"resnet_{split_name}_features.npy", features)

    # Only save y_* if missing, to avoid overwriting BERT labels accidentally
    y_path = PROBS_DIR / f"y_{split_name}.npy"
    if not y_path.exists():
        np.save(y_path, labels)

    print(split_name, "probs", probs.shape, "features", features.shape, "labels", labels.shape)

train probs (480, 4) features (480, 512) labels (480,)
val probs (103, 4) features (103, 512) labels (103,)
test probs (104, 4) features (104, 512) labels (104,)


In [16]:
!ls "/content/drive/MyDrive/Movie_Genre_Project/outputs/probs/y_test.npy"

/content/drive/MyDrive/Movie_Genre_Project/outputs/probs/y_test.npy


In [18]:
base = "/content/drive/MyDrive/Movie_Genre_Project/outputs/probs"

print("BERT:", np.load(f"{base}/bert_test_probs.npy").shape)
print("ResNet:", np.load(f"{base}/resnet_test_probs.npy").shape)
print("y_test:", np.load(f"{base}/y_test.npy").shape)

BERT: (104, 4)
ResNet: (104, 4)
y_test: (104,)


In [19]:
from pathlib import Path
import json
import numpy as np

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, f1_score, classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline


PROJECT_DIR = Path("/content/drive/MyDrive/Movie_Genre_Project")
OUTPUT_DIR = PROJECT_DIR / "outputs"
PROBS_DIR = OUTPUT_DIR / "probs"
FEATURES_DIR = OUTPUT_DIR / "features"
METRICS_DIR = OUTPUT_DIR / "metrics"
METRICS_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["action", "comedy", "horror", "romance"]


def evaluate(name, y_true, preds):
    acc = accuracy_score(y_true, preds)
    macro_f1 = f1_score(y_true, preds, average="macro")

    print(f"\n{name}")
    print(f"Accuracy: {acc:.4f}")
    print(f"Macro F1: {macro_f1:.4f}")
    print(classification_report(y_true, preds, target_names=CLASS_NAMES, zero_division=0))
    print("Confusion Matrix:")
    print(confusion_matrix(y_true, preds))

    return {
        "model": name,
        "accuracy": float(acc),
        "macro_f1": float(macro_f1),
        "classification_report": classification_report(
            y_true,
            preds,
            target_names=CLASS_NAMES,
            output_dict=True,
            zero_division=0,
        ),
        "confusion_matrix": confusion_matrix(y_true, preds).tolist(),
    }


# -------------------------
# Load probability outputs
# -------------------------
bert_test_probs = np.load(PROBS_DIR / "bert_test_probs.npy")
resnet_test_probs = np.load(PROBS_DIR / "resnet_test_probs.npy")
y_test = np.load(PROBS_DIR / "y_test.npy", allow_pickle=True)

print("Loaded test probabilities:")
print("BERT:", bert_test_probs.shape)
print("ResNet:", resnet_test_probs.shape)
print("y_test:", y_test.shape)

if bert_test_probs.shape != resnet_test_probs.shape:
    raise ValueError(f"Shape mismatch: BERT {bert_test_probs.shape}, ResNet {resnet_test_probs.shape}")

if bert_test_probs.shape[0] != y_test.shape[0]:
    raise ValueError(f"Label mismatch: probs {bert_test_probs.shape[0]}, labels {y_test.shape[0]}")

results = []


# -------------------------
# Late Fusion
# -------------------------
print("\n==============================")
print("LATE FUSION RESULTS")
print("==============================")

best_late = None

for w_bert in [0.5, 0.6, 0.7, 0.8, 0.9]:
    w_resnet = 1.0 - w_bert

    fused_probs = (w_bert * bert_test_probs) + (w_resnet * resnet_test_probs)
    fused_preds = fused_probs.argmax(axis=1)

    row = evaluate(
        f"Late Fusion BERT {w_bert:.1f} / ResNet {w_resnet:.1f}",
        y_test,
        fused_preds,
    )

    results.append(row)

    if best_late is None or row["macro_f1"] > best_late["macro_f1"]:
        best_late = row


# -------------------------
# Intermediate Fusion
# -------------------------
print("\n==============================")
print("INTERMEDIATE FUSION RESULTS")
print("==============================")

X_train_bert = np.load(FEATURES_DIR / "bert_train_embeddings.npy")
X_test_bert = np.load(FEATURES_DIR / "bert_test_embeddings.npy")

X_train_resnet = np.load(FEATURES_DIR / "resnet_train_features.npy")
X_test_resnet = np.load(FEATURES_DIR / "resnet_test_features.npy")

y_train = np.load(PROBS_DIR / "y_train.npy", allow_pickle=True)

print("BERT train embeddings:", X_train_bert.shape)
print("ResNet train features:", X_train_resnet.shape)
print("BERT test embeddings:", X_test_bert.shape)
print("ResNet test features:", X_test_resnet.shape)
print("y_train:", y_train.shape)

X_train_fused = np.concatenate([X_train_bert, X_train_resnet], axis=1)
X_test_fused = np.concatenate([X_test_bert, X_test_resnet], axis=1)

clf = make_pipeline(
    StandardScaler(),
    LogisticRegression(max_iter=3000, class_weight="balanced")
)

clf.fit(X_train_fused, y_train)
intermediate_preds = clf.predict(X_test_fused)

intermediate_row = evaluate(
    "Intermediate Fusion BERT Embeddings + ResNet Features",
    y_test,
    intermediate_preds,
)

results.append(intermediate_row)


# -------------------------
# Save results
# -------------------------
save_path = METRICS_DIR / "week3_fusion_results.json"

with open(save_path, "w") as f:
    json.dump(results, f, indent=2)

print("\n==============================")
print("SUMMARY")
print("==============================")

for row in results:
    print(f"{row['model']}: Accuracy={row['accuracy']:.4f}, Macro F1={row['macro_f1']:.4f}")

print("\nBest late fusion:")
print(f"{best_late['model']}: Accuracy={best_late['accuracy']:.4f}, Macro F1={best_late['macro_f1']:.4f}")

print("\nSaved results to:")
print(save_path)

Loaded test probabilities:
BERT: (104, 4)
ResNet: (104, 4)
y_test: (104,)

LATE FUSION RESULTS

Late Fusion BERT 0.5 / ResNet 0.5
Accuracy: 0.8365
Macro F1: 0.7993
              precision    recall  f1-score   support

      action       0.88      0.83      0.86        36
      comedy       0.69      0.69      0.69        16
      horror       0.84      0.97      0.90        38
     romance       0.90      0.64      0.75        14

    accuracy                           0.84       104
   macro avg       0.83      0.78      0.80       104
weighted avg       0.84      0.84      0.83       104

Confusion Matrix:
[[30  2  4  0]
 [ 2 11  2  1]
 [ 1  0 37  0]
 [ 1  3  1  9]]

Late Fusion BERT 0.6 / ResNet 0.4
Accuracy: 0.8558
Macro F1: 0.8203
              precision    recall  f1-score   support

      action       0.91      0.86      0.89        36
      comedy       0.65      0.81      0.72        16
      horror       0.90      0.95      0.92        38
     romance       0.90      0.64   